# Elevation Grid: Downsampled Height Lookup

Creates a compact (~500 KB) elevation grid covering the Oslo bike-sharing area from the DOM1 GeoTIFF tiles.
Given any lat/lon, the grid can return an approximate elevation without loading 4 GB of raster data.

Output: `prepared-data/elevation_grid.json`

## Setup & Imports

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd()
project_root = cwd if (cwd / "package.json").exists() else cwd.parent.parent
prepared_dir = project_root / "prepared-data"
dom1_dir = project_root / "dom1"

sys.path.insert(0, str(project_root / "data-pipeline"))
from execution_utils import show_execution_banner, write_with_execution_metadata
from elevation_utils import build_tile_index, sample_elevations_bulk, latlon_to_utm

print("Project root:", project_root)

out_path = prepared_dir / "elevation_grid.json"
_pipeline_start_time = show_execution_banner(out_path)

## Define Grid Parameters

In [ ]:
# Oslo bike-sharing area bounds (WGS84) with padding
MIN_LAT, MAX_LAT = 59.85, 59.98
MIN_LON, MAX_LON = 10.60, 10.90
CELL_SIZE_M = 50  # meters

# Convert corners to UTM to define the grid
e_min, n_min = latlon_to_utm(MIN_LAT, MIN_LON)
e_max, n_max = latlon_to_utm(MAX_LAT, MAX_LON)

# Snap to cell_size boundaries
origin_e = np.floor(e_min / CELL_SIZE_M) * CELL_SIZE_M
origin_n = np.ceil(n_max / CELL_SIZE_M) * CELL_SIZE_M  # top-left (max northing)

ncols = int(np.ceil((e_max - origin_e) / CELL_SIZE_M))
nrows = int(np.ceil((origin_n - n_min) / CELL_SIZE_M))

print(f"UTM bounds: E=[{e_min:.0f}, {e_max:.0f}], N=[{n_min:.0f}, {n_max:.0f}]")
print(f"Grid: {ncols} cols x {nrows} rows = {ncols * nrows:,} cells")
print(f"Origin (upper-left): E={origin_e:.0f}, N={origin_n:.0f}")

## Sample Elevations

In [ ]:
tile_index = build_tile_index(dom1_dir)
print(f"Loaded {len(tile_index)} tiles")

# Build coordinate arrays for all grid cell centers
col_idx = np.arange(ncols)
row_idx = np.arange(nrows)
cc, rr = np.meshgrid(col_idx, row_idx)

eastings = origin_e + (cc.ravel() + 0.5) * CELL_SIZE_M
northings = origin_n - (rr.ravel() + 0.5) * CELL_SIZE_M

print(f"Sampling {len(eastings):,} points...")
heights = sample_elevations_bulk(eastings, northings, tile_index)

valid = ~np.isnan(heights)
print(f"Valid: {valid.sum():,} / {len(heights):,}")
print(f"Elevation range: {np.nanmin(heights):.1f} - {np.nanmax(heights):.1f} m")

height_grid = heights.reshape(nrows, ncols)

## Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(height_grid, cmap="terrain", origin="upper",
               extent=[origin_e, origin_e + ncols * CELL_SIZE_M,
                       origin_n - nrows * CELL_SIZE_M, origin_n])
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_title(f"Oslo Elevation Grid ({CELL_SIZE_M}m resolution)")
plt.colorbar(im, ax=ax, label="Elevation (m)", shrink=0.8)
plt.tight_layout()
plt.show()

## Write Output

In [ ]:
# Convert to list of lists, rounding to 1 decimal. Replace NaN with None.
heights_list = []
for row in height_grid:
    heights_list.append([round(float(v), 1) if not np.isnan(v) else None for v in row])

grid_data = {
    "crs": "EPSG:25833",
    "origin_easting": float(origin_e),
    "origin_northing": float(origin_n),
    "cell_size_m": CELL_SIZE_M,
    "nrows": nrows,
    "ncols": ncols,
    "wgs84_bounds": {
        "min_lat": MIN_LAT,
        "max_lat": MAX_LAT,
        "min_lon": MIN_LON,
        "max_lon": MAX_LON,
    },
    "heights": heights_list,
}

write_with_execution_metadata(out_path, grid_data, _pipeline_start_time)

size_kb = out_path.stat().st_size / 1024
print(f"Wrote {out_path} ({size_kb:.0f} KB)")

## Verify: Lookup Test

In [ ]:
import json
from elevation_utils import lookup_elevation_from_grid

with open(out_path, encoding="utf-8") as f:
    loaded = json.load(f)

# Test a few known locations
test_points = [
    ("Aker Brygge (near sea level)", 59.9113, 10.7276),
    ("Holmenkollen (high point)", 59.9636, 10.6674),
    ("Oslo S (central)", 59.9108, 10.7529),
]

for name, lat, lon in test_points:
    elev = lookup_elevation_from_grid(lat, lon, loaded["data"])
    print(f"{name}: {elev} m")